In [1]:
# The following lines enable automatic reloading of modules in an IPython/Jupyter environment.
# They work exactly like the commented lines below, but avoid errors when not running in such an environment.
# %load_ext autoreload
# %autoreload 2

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except (NameError, AttributeError):
    pass


In [2]:
answer = {}


In [3]:
import random
from pprint import pprint

import json_tricks
import matplotlib
import numpy as np
import torch
import torch.nn.functional as F

matplotlib.use('Agg')
import matplotlib.pyplot as plt

import sys
root_path = '../../../..'
sys.path.append(root_path)

import dotenv
dotenv.load_dotenv(dotenv.find_dotenv(root_path + '/.env'))

import src
from src import utils


# Word2Vec With Hierarchical Softmax

Word2Vec learns a vector for each word by asking a simple question again and again: given one word, which words tend to appear nearby?

In this task, we will implement a skip-gram Word2Vec model. The encoder is intentionally simple: one embedding matrix. The decoder is hierarchical softmax: for each target word, the model predicts a sequence of left/right decisions in a deterministic binary-index tree. Each decision is a sigmoid binary classifier attached to one internal tree node.

The result is still a fully connected model in spirit, but the output layer is much cheaper than a full vocabulary softmax when the vocabulary becomes large.

# Task 0: Prepare the Environment

## Goal

Set all random seeds with `src.utils.seed.seed_all(0)`. Word embeddings are sensitive to initialization and data order, so deterministic runs make debugging much easier.

## Implementation Details

Write the seed call in the cell below. The expected student code should be placed after `## YOUR CODE HERE`.


In [4]:
## YOUR CODE HERE
src.utils.seed.seed_all(0)

seed_check = {
    'python.random': [random.random() for _ in range(4)],
    'numpy.random.rand': np.random.rand(2, 3),
    'torch.rand': torch.rand(2, 3),
    'torch.backends.cudnn.deterministic': torch.backends.cudnn.deterministic,
}
answer['seed_check'] = seed_check
pprint(seed_check)


{'numpy.random.rand': array([[0.5488135 , 0.71518937, 0.60276338],
       [0.54488318, 0.4236548 , 0.64589411]]),
 'python.random': [0.8444218515250481,
                   0.7579544029403025,
                   0.420571580830845,
                   0.25891675029296335],
 'torch.backends.cudnn.deterministic': False,
 'torch.rand': tensor([[0.4963, 0.7682, 0.0885],
        [0.1320, 0.3074, 0.6341]])}


# Task 1: Build the Skip-Gram Dataset

## Goal

Edit [src/datasets/wikicorpora.py](../../../../src/datasets/wikicorpora.py) and implement `SkipGramDataset`.

The dataset should turn a text corpus into skip-gram training examples. The notebook trains on a larger corpus source: cached WikiText-2 when it is already available locally, otherwise a deterministic larger toy corpus. The dataset should not know anything about hierarchical softmax. Its job is to tokenize text, build a vocabulary, and return word-index pairs.

## What The Dataset Must Build

Inside `__init__`, create:
- `self.tokens`: normalized alphabetic tokens from the corpus. Token normalization has two steps: first lowercase the token, then cut common suffixes such as `-ing`, `-ed`, `-es`, and `-s` so related forms like `play`, `played`, and `playing` land closer together (but note that the remaining word should contain at least 2 letters);
- `self.vocab`: words with frequency at least `min_count`, sorted by decreasing frequency and then alphabetically;
- `self.word_to_index` and `self.index_to_word`;
- `self.word_counts`: counts for words that remain in the vocabulary;
- `self.pairs`: `(center_word, context_word)` index pairs for every word within `window_size` positions.

## What `__getitem__` Must Return

Return a dictionary with only:
- `center_word`: integer index of the input word in the vocabulary;
- `context_word`: integer index of the target nearby word in the vocabulary.

In [5]:
def build_large_toy_corpus(repetitions=12):
    topic_lines = [
        'king queen prince princess palace crown royal throne kingdom noble court',
        'queen palace crown royal ceremony crown palace kingdom court noble',
        'cat dog pet animal home garden friendly playful sleeping walking',
        'dog cat animal pet family home walking playing running resting',
        'apple orange banana fruit sweet tree garden market fresh basket',
        'orange apple banana fruit juice sweet fresh market basket tree',
        'science experiment theory data model research laboratory method result',
        'scientist researcher laboratory experiment data method model theory result',
        'music melody rhythm song singer guitar piano concert stage audience',
        'painter painting color canvas museum artist gallery portrait sketch',
        'river ocean water island mountain forest valley weather cloud storm',
        'teacher student lesson school classroom book reading writing learning',
        'cat kitten dog pet animal home garden playful friendly',
        'cat kitten pet animal furry playful home friendly',
        'cucumber tomato vegetable salad garden fresh green food',
        'cucumber vegetable salad tomato fresh green market garden',
        'london england britain city capital river museum street',
        'london city england capital street bridge river museum',
    ]
    corpus = []
    for repetition in range(repetitions):
        for line in topic_lines:
            corpus.append(line)
    return corpus


def demo_anchor_corpus():
    return [
        'cat kitten dog pet animal home garden playful friendly',
        'cat kitten pet animal furry playful home friendly',
        'cucumber tomato vegetable salad garden fresh green food',
        'cucumber vegetable salad tomato fresh green market garden',
        'london england britain city capital river museum street',
        'london city england capital street bridge river museum',
    ]


def load_training_corpus(max_wikitext_lines=120):
    try:
        from datasets import load_dataset
        wiki = load_dataset(
            'wikitext',
            'wikitext-2-raw-v1',
            split='train',
            download_mode='reuse_dataset_if_exists',
        )
        lines = [
            row['text'].strip()
            for row in wiki
            if row['text'].strip() and not row['text'].strip().startswith('=')
        ][:max_wikitext_lines]
        if len(lines) >= 30:
            return lines + demo_anchor_corpus(), 'cached WikiText-2 plus demo anchors'
    except Exception:
        pass
    return build_large_toy_corpus(), 'deterministic large toy corpus'


training_corpus, corpus_source = load_training_corpus()

word_dataset = src.datasets.wikicorpora.SkipGramDataset(
    training_corpus,
    window_size=2,
    min_count=2,
)

suffix_check_dataset = src.datasets.wikicorpora.SkipGramDataset(
    ['playing played plays foxes dogs princess'],
    window_size=1,
    min_count=1,
)

print('corpus source:', corpus_source)
print('corpus lines:', len(training_corpus))
print('vocab size:', len(word_dataset.vocab))
print('num pairs:', len(word_dataset))
print('first sample:')
pprint(word_dataset[0])
print('suffix-normalized tokens:', suffix_check_dataset.tokens)

answer['corpus_source'] = corpus_source
answer['training_corpus_length'] = len(training_corpus)
answer['vocab'] = word_dataset.vocab
answer['word_to_index'] = word_dataset.word_to_index
answer['word_counts'] = word_dataset.word_counts
answer['dataset_length'] = len(word_dataset)
answer['first_sample'] = word_dataset[0]
answer['suffix_normalized_tokens'] = suffix_check_dataset.tokens


corpus source: deterministic large toy corpus
corpus lines: 216
vocab size: 100
num pairs: 7962
first sample:
{'center_word': tensor(64), 'context_word': tensor(34)}
suffix-normalized tokens: ['play', 'play', 'play', 'fox', 'dog', 'princess']


# Task 2: Build Binary-Index Tree Targets

## Goal

Edit [src/models/feedforward/word2vec.py](../../../../src/models/feedforward/word2vec.py) and implement `BinaryIndexTree.__call__`.

Hierarchical softmax replaces one large vocabulary softmax with several binary decisions. In this notebook, the decisions come from a deterministic complete binary tree: the target word index is written as a fixed-width binary string, and each bit tells the model whether to go left (`0`) or right (`1`).

## How The Binary Tree Works

The tree is a complete binary tree with heap-style internal node ids:
- root node: `0`;
- left child of node `n`: `2 * n + 1`;
- right child of node `n`: `2 * n + 2`.

Every word index is converted into a fixed-width binary target string. The width is `self.depth = max(1, (vocab_size - 1).bit_length())`, so every word receives the same number of binary decisions. This avoids a special case for word index `0`: it simply becomes all-left decisions.

For example, with `vocab_size = 6`, `self.depth = 3`:
- word index `0` becomes binary targets `000`;
- word index `5` becomes binary targets `101`.

The `targets` are the binary decisions the model should predict. A target bit of `0` means left, and a target bit of `1` means right.

The `path` contains the internal node where each decision is made. For decision step `s`, take the prefix before that decision and convert it to a node id:
- empty prefix `""` means the root node `0`;
- for a nonempty prefix with length `d` and binary value `p`, the node id is `(2 ** d - 1) + p`.

So for word index `5` in a vocabulary of size `6`:
- fixed-width binary targets: `101`;
- decision 0 uses prefix `""`, so node `0`, target `1`;
- decision 1 uses prefix `"1"`, so node `(2 ** 1 - 1) + 1 = 2`, target `0`;
- decision 2 uses prefix `"10"`, so node `(2 ** 2 - 1) + 2 = 5`, target `1`;
- final `path = [0, 2, 5]` and `targets = [1, 0, 1]`.

For word index `0`:
- fixed-width binary targets: `000`;
- prefixes are `""`, `"0"`, and `"00"`;
- final `path = [0, 1, 3]` and `targets = [0, 0, 0]`.

The `mask` is part of the shared hierarchical-softmax batch contract. In this deterministic fixed-depth tree, every path has length `self.depth`, so the mask is all ones with shape `(batch_size, self.max_path_length)`.

## Implementation Details

The class already gives you helper methods:
- `self.depth`: number of binary decisions for each word. It equals `(vocab_size - 1).bit_length()`, with a minimum of `1`;
- `self.max_path_length`: same value as `self.depth`;
- `self.num_internal_nodes`: number of internal nodes in the complete tree, `2 ** self.depth - 1`;
- `self.targets_for_index(word_index)`: fixed-width binary string for the word index;
- `self.node_id_from_prefix(prefix_bits)`: heap-style internal node id for a binary prefix;
- `self.path_and_targets(word_index)`: Python lists containing the node path and binary targets for one word.

Implement `__call__(context_word)` so it works on a 1D tensor, or on any tensor that can be flattened into word indices. The method should:
- remember `device = context_word.device`;
- detach, move to CPU, flatten, and convert the indices to a Python list;
- for every word index, call `self.path_and_targets(word_index)`;
- collect all paths into a tensor named `path` with dtype `torch.long` on the original device;
- collect all binary targets into a tensor named `targets` with dtype `torch.float32` on the original device;
- create `mask`, a tensor of ones with shape `(batch_size, self.max_path_length)`, dtype `torch.float32`, on the original device;
- return exactly `{"path": path, "targets": targets, "mask": mask}`.

The fallback implementation returns correctly shaped zero paths and zero targets, so the notebook will still run before you solve the task. It will not pass the checks because the tree routes will be wrong.


In [6]:
binary_tree = src.models.feedforward.word2vec.BinaryIndexTree(vocab_size=6)
visible_words = torch.tensor([0, 5])
visible_result = binary_tree(visible_words)
visible_expected_path = torch.tensor([[0, 1, 3], [0, 2, 5]])
visible_expected_targets = torch.tensor([[0., 0., 0.], [1., 0., 1.]])
visible_expected_mask = torch.ones(2, 3)

print('visible words:', visible_words.tolist())
print('expected path:')
print(visible_expected_path)
print('actual path:')
print(visible_result['path'])
print('expected targets:')
print(visible_expected_targets)
print('actual targets:')
print(visible_result['targets'])

binary_tree_visible_ok = (
    torch.equal(visible_result['path'].cpu(), visible_expected_path)
    and torch.equal(visible_result['targets'].cpu(), visible_expected_targets)
    and torch.equal(visible_result['mask'].cpu(), visible_expected_mask)
)
print('visible check passed:', binary_tree_visible_ok)

check_tree = src.models.feedforward.word2vec.BinaryIndexTree(vocab_size=8)
check_words = torch.tensor([1, 2, 7])
check_result = check_tree(check_words)

answer['binary_tree_depth'] = check_tree.depth
answer['binary_tree_num_internal_nodes'] = check_tree.num_internal_nodes
answer['binary_tree_visible_ok'] = binary_tree_visible_ok
answer['binary_tree_check_words'] = check_words
answer['binary_tree_check_path'] = check_result['path']
answer['binary_tree_check_targets'] = check_result['targets']
answer['binary_tree_check_mask'] = check_result['mask']


visible words: [0, 5]
expected path:
tensor([[0, 1, 3],
        [0, 2, 5]])
actual path:
tensor([[0, 1, 3],
        [0, 2, 5]])
expected targets:
tensor([[0., 0., 0.],
        [1., 0., 1.]])
actual targets:
tensor([[0., 0., 0.],
        [1., 0., 1.]])
visible check passed: True


# Task 3: Build the Hierarchical-Softmax Layer

## Goal

Edit [src/models/feedforward/word2vec.py](../../../../src/models/feedforward/word2vec.py) and implement `HierarchicalSoftmax.__init__` and `HierarchicalSoftmax.forward`.

The previous task built the deterministic binary tree. This task turns that tree into a neural output layer: each internal tree node has a trainable vector, and each vector asks one binary question about where the target word is located.

## Constructor

Inside `__init__(self, embedding_dim, vocab_size)`, create:
- `self.embedding_dim`: integer embedding dimension;
- `self.targets`: a `BinaryIndexTree(vocab_size)` object;
- `self.decoder`: a `torch.nn.Embedding(self.targets.num_internal_nodes, self.embedding_dim)` layer;
- initialize `self.decoder.weight` with `torch.nn.init.normal_(..., mean=0.0, std=0.02)`.

The fallback constructor creates the same shape but fills the decoder with zeros. That keeps the notebook runnable, but a zero decoder cannot learn useful route probabilities.

## Forward Pass

The method receives:
- `embedding`: center-word vectors with shape `(batch_size, embedding_dim)`;
- `target_word`: target context-word indices with shape `(batch_size,)` or any flattenable shape.

Implement `forward` so it:
- calls `self.targets(target_word)` to get `path`, `targets`, and `mask`;
- uses `self.decoder(path)` to get one node vector per binary decision;
- computes `logits` as dot products between each center-word embedding and each node vector, with shape `(batch_size, max_path_length)`;
- computes `probabilities = torch.sigmoid(logits)`;
- computes `target_probabilities`: use `probabilities` where the binary target is `1`, and `1 - probabilities` where the target is `0`;
- computes `total_probability` as the product of target probabilities along the path;
- computes binary cross entropy with logits against `targets`, with `reduction="none"`;
- multiplies per-node losses by `mask`, sums over the path into `per_word_loss`, and averages into scalar `loss`;
- returns a dictionary containing the target tensors plus `logits`, `probabilities`, `target_probabilities`, `total_probability`, `per_node_loss`, `per_word_loss`, and `loss`.

The fallback forward pass returns neutral probabilities and a valid loss, so the notebook can keep running before the method is solved. It will not pass the checks because it ignores the decoder vectors.


In [7]:
hsoftmax_demo = src.models.feedforward.word2vec.HierarchicalSoftmax(embedding_dim=2, vocab_size=4)
with torch.no_grad():
    hsoftmax_demo.decoder.weight.copy_(torch.tensor([
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]))

visible_embedding = torch.tensor([[1.0, 2.0], [0.5, -1.0]])
visible_target_word = torch.tensor([0, 3])
visible_hsoftmax = hsoftmax_demo(visible_embedding, visible_target_word)
visible_expected_path = torch.tensor([[0, 1], [0, 2]])
visible_expected_targets = torch.tensor([[0., 0.], [1., 1.]])
visible_expected_logits = torch.tensor([[1.0, 2.0], [0.5, -0.5]])
visible_expected_probabilities = torch.sigmoid(visible_expected_logits)
visible_expected_target_probabilities = torch.where(
    visible_expected_targets.bool(),
    visible_expected_probabilities,
    1.0 - visible_expected_probabilities,
)
visible_expected_total_probability = visible_expected_target_probabilities.prod(dim=1)

print('expected path:')
print(visible_expected_path)
print('actual path:')
print(visible_hsoftmax['path'])
print('expected logits:')
print(visible_expected_logits)
print('actual logits:')
print(visible_hsoftmax['logits'])
print('expected total probability:', visible_expected_total_probability)
print('actual total probability:', visible_hsoftmax['total_probability'])

hsoftmax_visible_ok = (
    torch.equal(visible_hsoftmax['path'].cpu(), visible_expected_path)
    and torch.equal(visible_hsoftmax['targets'].cpu(), visible_expected_targets)
    and torch.allclose(visible_hsoftmax['logits'].detach().cpu(), visible_expected_logits)
    and torch.allclose(visible_hsoftmax['total_probability'].detach().cpu(), visible_expected_total_probability)
)
print('visible check passed:', hsoftmax_visible_ok)

hsoftmax_check = src.models.feedforward.word2vec.HierarchicalSoftmax(embedding_dim=3, vocab_size=5)
with torch.no_grad():
    hsoftmax_check.decoder.weight.copy_(torch.arange(21, dtype=torch.float32).reshape(7, 3) / 10.0)
check_embedding = torch.tensor([[0.2, -0.4, 0.6], [1.0, 0.0, -1.0]])
check_target_word = torch.tensor([2, 4])
check_hsoftmax = hsoftmax_check(check_embedding, check_target_word)

answer['hsoftmax_visible_ok'] = hsoftmax_visible_ok
answer['hsoftmax_check_path'] = check_hsoftmax['path']
answer['hsoftmax_check_targets'] = check_hsoftmax['targets']
answer['hsoftmax_check_logits'] = check_hsoftmax['logits']
answer['hsoftmax_check_probabilities'] = check_hsoftmax['probabilities']
answer['hsoftmax_check_total_probability'] = check_hsoftmax['total_probability']
answer['hsoftmax_check_loss'] = check_hsoftmax['loss']


expected path:
tensor([[0, 1],
        [0, 2]])
actual path:
tensor([[0, 1],
        [0, 2]])
expected logits:
tensor([[ 1.0000,  2.0000],
        [ 0.5000, -0.5000]])
actual logits:
tensor([[ 1.0000,  2.0000],
        [ 0.5000, -0.5000]], grad_fn=<ViewBackward0>)
expected total probability: tensor([0.0321, 0.2350])
actual total probability: tensor([0.0321, 0.2350], grad_fn=<ProdBackward1>)
visible check passed: True


# Task 4: Build the Word2Vec Network

## Goal

Edit [src/models/feedforward/word2vec.py](../../../../src/models/feedforward/word2vec.py) and implement `Word2Vec.__init__` and `Word2Vec.forward`.

The previous task built the hierarchical-softmax output layer. This task wraps it with the Word2Vec encoder so the whole model follows the shared course batch contract.

## Constructor

Inside `__init__(self, vocab_size, embedding_dim)`, create:
- `self.vocab_size`: integer vocabulary size;
- `self.embedding_dim`: integer embedding dimension;
- `self.encoder`: a `torch.nn.Embedding(self.vocab_size, self.embedding_dim)` layer for center-word vectors;
- `self.hierarchical_softmax`: a `HierarchicalSoftmax(self.embedding_dim, self.vocab_size)` layer;
- `self.decoder`: an alias to `self.hierarchical_softmax.decoder`;
- `self.num_internal_nodes`: an alias to `self.hierarchical_softmax.num_internal_nodes`;
- initialize `self.encoder.weight` with `torch.nn.init.normal_(..., mean=0.0, std=0.02)`.

The fallback constructor creates the same modules but fills the encoder with zeros. That keeps the notebook runnable, but a zero encoder cannot learn meaningful word vectors.

## Forward Pass

The training loop passes a nested batch dictionary:

```python
batch = {'data': original_batch}
```

Implement `forward(self, batch)` so it mutates and returns the same batch dictionary:
- read `center_word = batch['data']['center_word']`;
- compute `embedding = self.encoder(center_word)`;
- create `batch['signals'] = {'embedding': embedding}`;
- create `batch['postprocessed'] = {}`;
- if `context_word` is present in `batch['data']`, call `self.hierarchical_softmax(embedding, batch['data']['context_word'])`;
- copy `path`, `targets`, and `mask` from the hierarchical-softmax output into `batch['data']`;
- copy `logits`, `probabilities`, `target_probabilities`, `total_probability`, and `loss` into `batch['signals']`;
- store predicted binary decisions in `batch['postprocessed']['targets']` by thresholding probabilities at `0.5`;
- return `batch`.

The fallback forward pass fills the batch with neutral zero-logit predictions, so the notebook can still reach later cells before the method is solved. It will not pass the checks because it ignores the learned encoder and decoder interaction.


In [8]:
word2vec_demo = src.models.feedforward.word2vec.Word2Vec(
    vocab_size=4,
    embedding_dim=2,
)
with torch.no_grad():
    word2vec_demo.encoder.weight.copy_(torch.tensor([
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
        [-1.0, 0.5],
    ]))
    word2vec_demo.decoder.weight.copy_(torch.tensor([
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]))

visible_batch = {'data': {'center_word': torch.tensor([0, 3]), 'context_word': torch.tensor([0, 3])}}
visible_result = word2vec_demo(visible_batch)
visible_expected_embedding = torch.tensor([[1.0, 0.0], [-1.0, 0.5]])
visible_expected_path = torch.tensor([[0, 1], [0, 2]])
visible_expected_targets = torch.tensor([[0., 0.], [1., 1.]])
visible_expected_logits = torch.tensor([[1.0, 0.0], [-1.0, -0.5]])

print('expected embedding:')
print(visible_expected_embedding)
print('actual embedding:')
print(visible_result['signals']['embedding'])
print('expected logits:')
print(visible_expected_logits)
print('actual logits:')
print(visible_result['signals']['logits'])

word2vec_visible_ok = (
    torch.allclose(visible_result['signals']['embedding'].detach().cpu(), visible_expected_embedding)
    and torch.equal(visible_result['data']['path'].cpu(), visible_expected_path)
    and torch.equal(visible_result['data']['targets'].cpu(), visible_expected_targets)
    and torch.allclose(visible_result['signals']['logits'].detach().cpu(), visible_expected_logits)
)
print('visible check passed:', word2vec_visible_ok)

embedding_dim = 3

word2vec_model = src.models.feedforward.word2vec.Word2Vec(
    vocab_size=len(word_dataset.vocab),
    embedding_dim=embedding_dim,
)

check_batch = {
    'data': {
        key: value.unsqueeze(0) if torch.is_tensor(value) else value
        for key, value in word_dataset[0].items()
    }
}

with torch.no_grad():
    word2vec_model(check_batch)

print('embedding:', check_batch['signals']['embedding'].shape)
print('logits:', check_batch['signals']['logits'].shape)
print('num internal nodes:', word2vec_model.hierarchical_softmax.num_internal_nodes)
print('max path length:', word2vec_model.hierarchical_softmax.max_path_length)
pprint(check_batch)

answer['word2vec_visible_ok'] = word2vec_visible_ok
answer['word2vec_visible_embedding'] = visible_result['signals']['embedding']
answer['word2vec_visible_logits'] = visible_result['signals']['logits']
answer['check_batch'] = check_batch
answer['encoder_shape'] = list(word2vec_model.encoder.weight.shape)
answer['decoder_shape'] = list(word2vec_model.hierarchical_softmax.decoder.weight.shape)
answer['num_internal_nodes'] = word2vec_model.hierarchical_softmax.num_internal_nodes
answer['max_path_length'] = word2vec_model.hierarchical_softmax.max_path_length


expected embedding:
tensor([[ 1.0000,  0.0000],
        [-1.0000,  0.5000]])
actual embedding:
tensor([[ 1.0000,  0.0000],
        [-1.0000,  0.5000]], grad_fn=<EmbeddingBackward0>)
expected logits:
tensor([[ 1.0000,  0.0000],
        [-1.0000, -0.5000]])
actual logits:
tensor([[ 1.0000,  0.0000],
        [-1.0000, -0.5000]], grad_fn=<ViewBackward0>)
visible check passed: True
embedding: torch.Size([1, 3])
logits: torch.Size([1, 7])
num internal nodes: 127
max path length: 7
{'data': {'center_word': tensor([64]),
          'context_word': tensor([34]),
          'mask': tensor([[1., 1., 1., 1., 1., 1., 1.]]),
          'path': tensor([[ 0,  1,  4,  9, 19, 39, 80]]),
          'targets': tensor([[0., 1., 0., 0., 0., 1., 0.]])},
 'postprocessed': {'targets': tensor([[1, 0, 1, 1, 1, 1, 0]])},
 'signals': {'embedding': tensor([[-0.0185, -0.0041,  0.0208]]),
             'logits': tensor([[ 3.4103e-05, -1.7671e-04,  1.4098e-03,  1.6980e-04,  8.7237e-05,
          6.5657e-05, -6.0687e-04]]),

# Task 5: Create Hierarchical-Softmax Loss and Optimizer

## Goal

Create the components required by the shared training loop.

## Loss

The Word2Vec model now includes the hierarchical-softmax output layer. During the forward pass, that layer converts each `context_word` into binary-index-tree targets and returns one logit for each left/right decision along the target word's path.

Implement `hierarchical_softmax_loss(batch)`:
- read `logits` from `batch['signals']['logits']`;
- read binary `targets` and `mask` from `batch['data']`;
- compute binary cross entropy with logits against the `targets`;
- multiply by `mask` to keep the same contract as other hierarchical-softmax variants. In this deterministic fixed-depth tree, the mask is all ones;
- sum the masked losses along each path and return the average per word in the batch.

The loss is deliberately ordinary: hierarchical softmax is the output layer, and the loss simply trains the returned set of binary logits.

## Other Components

Create:
- `losses = {'hsoftmax': hierarchical_softmax_loss}`;
- `metrics = {}` because this task tracks loss and embeddings rather than classification accuracy;
- `optimizer = torch.optim.AdamW(word2vec_model.parameters(), lr=1.0e-2)`;
- `scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)`.


In [9]:
def hierarchical_softmax_loss(batch):
    return batch['signals']['embedding'].abs().mean()

losses = {'hsoftmax': hierarchical_softmax_loss}
metrics = {}
optimizer = torch.optim.AdamW(word2vec_model.parameters(), lr=1.0e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

## YOUR CODE HERE

with torch.no_grad():
    test_loss_val = {name: fn(check_batch) for name, fn in losses.items()}

answer['loss_names'] = list(losses.keys())
answer['metric_names'] = list(metrics.keys())
answer['optimizer_class'] = optimizer.__class__.__name__
answer['optimizer_defaults'] = dict(optimizer.defaults)
answer['scheduler_class'] = scheduler.__class__.__name__
answer['test_loss_val'] = test_loss_val
answer['hsoftmax_target_path_shape'] = list(check_batch['data']['path'].shape)
answer['hsoftmax_target_targets_shape'] = list(check_batch['data']['targets'].shape)
answer['hsoftmax_target_mask_shape'] = list(check_batch['data']['mask'].shape)
pprint(test_loss_val)


{'hsoftmax': tensor(0.0145)}


# Task 6: Create DataLoaders

## Goal

Create training and validation dataloaders from the skip-gram dataset.

The checker-friendly setup uses the same skip-gram examples for validation. With a real full-size corpus, you would split text into separate train and validation regions before building pairs.

## Implementation Details

Create:
- `batch_size = 64`;
- `train_dl` with `shuffle=True` and `drop_last=True`;
- `valid_dl` with `batch_size=batch_size` and `shuffle=False`.


In [10]:
batch_size = 64
train_dl = torch.utils.data.DataLoader(word_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
valid_dl = torch.utils.data.DataLoader(word_dataset, batch_size=batch_size, shuffle=False)
## YOUR CODE HERE

train_batch = next(iter(train_dl))
valid_batch = next(iter(valid_dl))

pprint(train_batch)
pprint(valid_batch)

answer['train_batch'] = str(train_batch)
answer['valid_batch'] = str(valid_batch)


{'center_word': tensor([46, 25,  4,  2, 63, 35, 18, 24, 37,  8, 42, 33, 17, 51,  2, 11, 41, 41,
        26, 66, 27, 17, 72, 43, 26, 22,  8, 44, 93, 57, 47,  4,  9, 42, 35, 38,
        33, 20,  4, 84, 12, 79,  4, 21, 27, 31, 28, 16, 45, 16, 70, 32, 23, 11,
        46,  3,  0, 52,  1, 59, 96, 10, 13, 20]),
 'context_word': tensor([91, 20,  5, 99,  3, 80, 38, 39,  3,  4, 23, 15, 18,  6,  5, 51, 37,  8,
        95, 93,  8, 48, 91, 41, 32,  0, 41, 89, 94, 20, 99, 56,  3,  9, 28, 22,
        15, 57, 12, 52,  8, 30, 44, 40, 65, 30, 35, 33, 50,  3, 97, 26, 21, 76,
        55, 25, 29, 78,  0,  5, 53, 64, 17, 43])}
{'center_word': tensor([64, 64, 34, 34, 34, 76, 76, 76, 76, 77, 77, 77, 77, 11, 11, 11, 11,  6,
         6,  6,  6, 36, 36, 36, 36, 95, 95, 95, 95, 26, 26, 26, 26, 32, 32, 32,
        32, 19, 19, 19, 19, 34, 34, 34, 34, 11, 11, 11, 11,  6,  6,  6,  6, 36,
        36, 36, 36, 51, 51, 51, 51,  6,  6,  6]),
 'context_word': tensor([34, 76, 64, 76, 77, 64, 34, 77, 11, 34, 76, 11,  6, 76, 

# Task 7: Train With the Shared Training Loop

## Goal

Reuse `src.train_loop.train_model`, exactly as in the previous tasks.

The shared loop will:
- wrap dataloader output as `{'data': batch}`;
- call `word2vec_model(batch)`;
- compute the named hierarchical-softmax loss;
- backpropagate through the summed named losses;
- report progress in the progress bars;
- log losses to MLflow.

The important design point is that Word2Vec does not need a custom training loop. It only needs a model and loss that follow the same batch contract.

In [11]:
n_epochs = 3

training_logger = src.utils.mlflow.prepare_mlflow_logger('word2vec_hsoftmax_training')

training_history = src.train_loop.train_model(
    word2vec_model,
    n_epochs,
    train_dl,
    valid_dl,
    losses,
    optimizer,
    metrics=metrics,
    scheduler=scheduler,
    mlflow_logger=training_logger,
    run_name='word2vec_hsoftmax_training',
)

train_hsoftmax_history = training_history['train_loss']['hsoftmax']
valid_hsoftmax_history = training_history['valid_loss']['hsoftmax']

training_logger.end()

answer['train_hsoftmax_history'] = train_hsoftmax_history
answer['valid_hsoftmax_history'] = valid_hsoftmax_history


/usr/local/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/07/05 11:20:45 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$G

[Open MLflow run with graphs](http://127.0.0.1:5000/#/experiments/682338566444037820/runs/1f45242584ee44b0b137927203c2a528)

epochs:   0%|          | 0/3 [00:00<?, ?it/s]

batches:   0%|          | 0/124 [00:00<?, ?it/s]

# Embedding Visualization

Because we used three-dimensional embeddings, we can plot the learned word vectors directly. Related words should begin to move toward nearby regions, although the checker-friendly run is still much smaller than a real Word2Vec training job.

We also inspect nearest neighbors for `cat`, `cucumber`, and `London`. This is the practical Word2Vec idea in miniature: words used in similar contexts should end up with nearby embeddings.


In [12]:
with torch.no_grad():
    embeddings = word2vec_model.encoder.weight.detach().cpu()

fig = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
ax.scatter(embeddings[:, 0], embeddings[:, 1], embeddings[:, 2], s=32)
for index, word in word_dataset.index_to_word.items():
    ax.text(
        float(embeddings[index, 0]),
        float(embeddings[index, 1]),
        float(embeddings[index, 2]),
        word,
        fontsize=8,
    )
ax.set_title('learned 3D word embeddings')
plt.show()

def nearest_words(query_word, embeddings, word_to_index, index_to_word, top_k=10):
    normalized_query = word_dataset._normalize_token(query_word)
    if normalized_query not in word_to_index:
        return []
    query_index = word_to_index[normalized_query]
    normalized_embeddings = F.normalize(embeddings, dim=1)
    similarities = normalized_embeddings @ normalized_embeddings[query_index]
    similarities[query_index] = -torch.inf
    neighbor_indices = torch.topk(similarities, k=min(top_k, similarities.numel() - 1)).indices.tolist()
    return [index_to_word[index] for index in neighbor_indices]

nearest_word_queries = ['cat', 'cucumber', 'London']
nearest_word_demo = {
    query: nearest_words(query, embeddings, word_dataset.word_to_index, word_dataset.index_to_word, top_k=10)
    for query in nearest_word_queries
}

print('nearest words:')
pprint(nearest_word_demo)

answer['embedding_matrix'] = embeddings
answer['embedding_matrix_shape'] = list(embeddings.shape)
answer['nearest_word_demo'] = nearest_word_demo

json_tricks.dump(utils.torch_to_numpy(answer), '.answer.json')


nearest words:
{'London': ['britain',
            'researcher',
            'food',
            'orange',
            'furry',
            'bridge',
            'dog',
            'rest',
            'fruit',
            'result'],
 'cat': ['theory',
         'song',
         'school',
         'laboratory',
         'guitar',
         'singer',
         'storm',
         'king',
         'lesson',
         'noble'],
 'cucumber': ['scientist',
              'kingdom',
              'apple',
              'friendly',
              'queen',
              'princess',
              'water',
              'classroom',
              'paint',
              'painter']}


'{"seed_check": {"python.random": [0.8444218515250481, 0.7579544029403025, 0.420571580830845, 0.25891675029296335], "numpy.random.rand": {"__ndarray__": [[0.5488135039273248, 0.7151893663724195, 0.6027633760716439], [0.5448831829968969, 0.4236547993389047, 0.6458941130666561]], "dtype": "float64", "shape": [2, 3], "Corder": true}, "torch.rand": {"__ndarray__": [[0.49625658988952637, 0.7682217955589294, 0.08847743272781372], [0.13203048706054688, 0.30742281675338745, 0.6340786814689636]], "dtype": "float32", "shape": [2, 3], "Corder": true}, "torch.backends.cudnn.deterministic": false}, "corpus_source": "deterministic large toy corpus", "training_corpus_length": 216, "vocab": ["garden", "animal", "cat", "fresh", "home", "pet", "crown", "dog", "friendly", "market", "museum", "palace", "playful", "river", "apple", "banana", "basket", "capital", "city", "court", "cucumber", "data", "england", "experiment", "fruit", "green", "kingdom", "kitten", "laboratory", "london", "method", "model", "n

# Task 8: Experiment Time

After the checker-friendly run works, try larger experiments:
- use a local cached WikiText-2, Wikipedia dump, Project Gutenberg text, or another open corpus;
- increase `embedding_dim` from `3` to `32`, `64`, or `128`;
- compare hierarchical softmax with a full vocabulary softmax or negative sampling;
- inspect nearest neighbors in embedding space;
- try larger context windows.

The conceptual takeaway is that the dataset is just skip-gram word-index pairs. Hierarchical softmax is an output layer and objective that replaces one expensive full-vocabulary prediction with a sequence of cheap sigmoid decisions.


In [13]:
# Optional long-run area. Do not include this in checker-facing answers.

# larger_model = src.models.feedforward.word2vec.Word2Vec(
#     vocab_size=len(word_dataset.vocab),
#     embedding_dim=32,
# )
# larger_optimizer = torch.optim.AdamW(larger_model.parameters(), lr=1.0e-2)
# larger_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(larger_optimizer)
# larger_logger = src.utils.mlflow.prepare_mlflow_logger('word2vec_long_run')
# larger_history = src.train_loop.train_model(
#     larger_model,
#     100,
#     train_dl,
#     valid_dl,
#     losses,
#     larger_optimizer,
#     metrics=metrics,
#     scheduler=larger_scheduler,
#     mlflow_logger=larger_logger,
#     run_name='word2vec_long_run',
# )
# larger_logger.end()
